# Build a question Answering application over a graph Database 


In [1]:
# Wait a few minutes before connecting using these details. Instance status can be viewed at https://console.neo4j.io
NEO4J_URI="neo4j+s://831fc8c3.databases.neo4j.io"
NEO4J_USERNAME="831fc8c3"
NEO4J_PASSWORD="CcFP9kIdApgtVH_lAIroaNwV2jnyPcgNadnwlfhcb_Q"
NEO4J_DATABASE="831fc8c3"
AURA_INSTANCEID="831fc8c3"
AURA_INSTANCENAME="Free instance"


In [3]:
import os 
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD

In [6]:
from langchain_community.graphs import Neo4jGraph
graph= Neo4jGraph(url=NEO4J_URI,username=NEO4J_USERNAME,password=NEO4J_PASSWORD)

graph

In [10]:
## Dataset Moview
movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE (m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
  MERGE (p:Person {name:trim(director)})
  MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
  MERGE (p:Person {name:trim(actor)})
  MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
  MERGE (g:Genre {name:trim(genre)})
  MERGE (m)-[:IN_GENRE]->(g))

"""

In [12]:
movie_query

"\nLOAD CSV WITH HEADERS FROM\n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row\n\nMERGE (m:Movie{id:row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\nFOREACH (director in split(row.director, '|') |\n  MERGE (p:Person {name:trim(director)})\n  MERGE (p)-[:DIRECTED]->(m))\nFOREACH (actor in split(row.actors, '|') |\n  MERGE (p:Person {name:trim(actor)})\n  MERGE (p)-[:ACTED_IN]->(m))\nFOREACH (genre in split(row.genres, '|') |\n  MERGE (g:Genre {name:trim(genre)})\n  MERGE (m)-[:IN_GENRE]->(g))\n\n"

In [13]:
graph.query(movie_query)

[#D97B]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 1.0705976342514925s (Unable to retrieve routing information)
[#D97C]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-2-0172.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Transaction failed and will be retried in 1.9683509853024046s (Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-2-0172.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))))


[]

In [15]:
graph.refresh_schema() 
print(graph.schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [16]:
import os 
from dotenv import load_dotenv

load_dotenv() 
groq_api_key = os.getenv("GROQ_API_KEY") 


In [23]:
from langchain_groq import ChatGroq
llm=ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001757FB95390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017501FAA250>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [24]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain

# Add the allow_dangerous_requests flag here
chain = GraphCypherQAChain.from_llm(
    graph=graph, 
    llm=llm, 
    verbose=True,
    allow_dangerous_requests=True
)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000001757BD81A10>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updat

In [25]:
# Wrap the key and value in curly braces {}
response = chain.invoke({"query": "Who was the director of the movie Casino?"})

# Print the final result
print(response["result"])




> Entering new GraphCypherQAChain chain...


[#C43A]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 0.849753429329458s (Unable to retrieve routing information)


Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"})
RETURN p.name;


[#C667]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-2-0172.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Transaction failed and will be retried in 1.9758067667589816s (Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-2-0172.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))))


Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.
Martin Scorsese was the director of the movie *Casino*.


# Prompt using 

In [26]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001757FB95390>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017501FAA250>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [27]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain

# Add the allow_dangerous_requests flag here
chain = GraphCypherQAChain.from_llm(
    graph=graph, 
    llm=llm, 
    verbose=True,
    
    allow_dangerous_requests=True
)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000001757BD81A10>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updat

In [28]:
chain.schema

<bound method BaseModel.schema of <class 'langchain_neo4j.chains.graph_qa.cypher.GraphCypherQAChain'>>

In [29]:
examples= [
{
"question": "How many artists are there?",
"query": "MATCH (a: Person)-[:ACTED_IN]->(: Movie) RETURN count (DISTINCT a)",
},
{
"question": "Which actors played in the movie Casino?",
"query": "MATCH (m: Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name",
},
{
"question": "How many movies has Tom Hanks acted in?",
"query": "MATCH (a: Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m: Movie) RETURN count(m)",
}
]

In [32]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# 1. Fixed plural typo: changed {questions} to {question}
example_prompt = PromptTemplate.from_template(
    "User input: {question}\nCypher query: {query}"
)

# 2. Re-structured to correctly utilize both 'schema' and 'question'
prompt = FewShotPromptTemplate(
    examples=examples[:5],
    example_prompt=example_prompt,
    prefix=(
        "You are a Neo4j expert. Given an input question, create a syntactically "
        "very accurate Cypher Query based on the database schema below:\n\n"
        "Schema: {schema}"  # Added {schema} here so LangChain can inject it
    ),
    suffix="\n\nUser input: {question}\nCypher query: ",
    input_variables=["question", "schema"]  # Fixed typo from 'imput_variables'
)


In [33]:
prompt

FewShotPromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, examples=[{'question': 'How many artists are there?', 'query': 'MATCH (a: Person)-[:ACTED_IN]->(: Movie) RETURN count (DISTINCT a)'}, {'question': 'Which actors played in the movie Casino?', 'query': "MATCH (m: Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a) RETURN a.name"}, {'question': 'How many movies has Tom Hanks acted in?', 'query': "MATCH (a: Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m: Movie) RETURN count(m)"}], example_prompt=PromptTemplate(input_variables=['query', 'question'], input_types={}, partial_variables={}, template='User input: {question}\nCypher query: {query}'), suffix='\n\nUser input: {question}\nCypher query: ', prefix='You are a Neo4j expert. Given an input question, create a syntactically very accurate Cypher Query based on the database schema below:\n\nSchema: {schema}')

In [34]:
print(prompt.format(question="How many artists are there?", schema="foo"))

You are a Neo4j expert. Given an input question, create a syntactically very accurate Cypher Query based on the database schema below:

Schema: foo

User input: How many artists are there?
Cypher query: MATCH (a: Person)-[:ACTED_IN]->(: Movie) RETURN count (DISTINCT a)

User input: Which actors played in the movie Casino?
Cypher query: MATCH (m: Movie {title: 'Casino'})<-[:ACTED_IN]-(a) RETURN a.name

User input: How many movies has Tom Hanks acted in?
Cypher query: MATCH (a: Person {name: 'Tom Hanks'})-[:ACTED_IN]->(m: Movie) RETURN count(m)



User input: How many artists are there?
Cypher query: 


In [39]:
llm
chain=GraphCypherQAChain.from_llm(graph=graph, llm=llm, cypher_prompt=prompt, verbose=True,allow_dangerous_requests=True)


In [41]:
chain.invoke("Which actors played in the movie Casino?")



> Entering new GraphCypherQAChain chain...


[#C036]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.161.242', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Unable to retrieve routing information
Transaction failed and will be retried in 1.1250417833894266s (Unable to retrieve routing information)


Generated Cypher:
MATCH (m:Movie {title: 'Casino'})<-[:ACTED_IN]-(a:Person)
RETURN a.name



[#C037]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-2-0172.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))): OSError('No data')
Transaction failed and will be retried in 2.061271338097043s (Failed to read from defunct connection IPv4Address(('p-mt-2a41353b6b9a-2-0172.production-orch-0695.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.161.242', 7687))))


Full Context:
[{'a.name': 'Robert De Niro'}, {'a.name': 'Joe Pesci'}, {'a.name': 'Sharon Stone'}, {'a.name': 'James Woods'}]

> Finished chain.


{'query': 'Which actors played in the movie Casino?',
 'result': 'Robert De\u202fNiro, Joe\u202fPesci, Sharon\u202fStone, James\u202fWoods.'}